In [1]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/ScrapeTraining/riyasewana_vehicles.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ScrapeTraining/riyasewana_vehicles.csv'

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
import pandas as pd
import numpy as np


df["Price"] = df["Price"].astype(str)


df["Price"] = df["Price"].str.replace("Rs.", "", regex=False)
df["Price"] = df["Price"].str.replace(",", "", regex=False)


df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

print(df["Price"].head())
print(df["Price"].dtype)
print("Null values:", df["Price"].isna().sum())


In [ ]:
print(df.info())

In [ ]:
df[df['Price'] > 30000000]

In [ ]:
df.shape

In [ ]:
df[df["Price"].isnull()]

In [ ]:
df = df[(df["Price"] <= 30000000) | (df["Price"].isna())]
df.shape

In [ ]:
df.to_csv('/content/drive/MyDrive/ScrapeTraining/clean#1.csv')

In [ ]:
"""
Sri Lanka Used Vehicle Price Prediction Model
=============================================
Data Source: Riyasewana.com (clean_1__1_.csv)
Model: Gradient Boosting Regressor + Linear Time-Series Trend
"""

import pandas as pd
import numpy as np
import re
import joblib
import warnings
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

warnings.filterwarnings('ignore')




def load_data(filepath: str) -> pd.DataFrame:
    df = pd.read_csv('/content/drive/MyDrive/ScrapeTraining/clean#1.csv')
    print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
    return df



def normalize_model_name(model) -> str:
    """
    Fixes inconsistent casing in model names.
    Examples:
        'AXIO'       -> 'Axio'
        'vitz ksp90' -> 'Vitz Ksp90'
        'WAGON R'    -> 'Wagon R'
    """
    if pd.isna(model):
        return model
    model = str(model).strip()
    model = re.sub(r'\s+', ' ', model)   # collapse multiple spaces
    model = model.title()                 # Title Case
    return model


def clean_model_names(df: pd.DataFrame) -> pd.DataFrame:
    original_unique = df['Model'].nunique()
    df['Model_Normalized'] = df['Model'].apply(normalize_model_name)
    cleaned_unique = df['Model_Normalized'].nunique()
    print(f"Model names: {original_unique:,} unique → {cleaned_unique:,} unique "
          f"(reduced by {original_unique - cleaned_unique:,})")
    return df




def engineer_features(df: pd.DataFrame, current_year: int = 2026) -> pd.DataFrame:
    # Parse published date
    df['published date'] = pd.to_datetime(df['published date'], format='%m/%d/%Y', errors='coerce')
    df['pub_month'] = df['published date'].dt.month
    df['pub_week']  = df['published date'].dt.isocalendar().week.astype(int)

    # Vehicle age
    df['vehicle_age'] = current_year - df['Year']

    return df




def preprocess(df: pd.DataFrame):
    """
    Filters, imputes, encodes and returns X, y + fitted encoders.
    """
    # Keep only rows with price
    df = df.dropna(subset=['Price']).copy()

    # Remove price outliers
    df = df[(df['Price'] > 100_000) & (df['Price'] < 30_000_000)]
    print(f"After price filtering: {len(df):,} rows")

    # Impute mileage with per-model median, then global median
    df['Milleage'] = df['Milleage'].fillna(
        df.groupby('Model_Normalized')['Milleage'].transform('median')
    )
    df['Milleage'] = df['Milleage'].fillna(df['Milleage'].median())

    # Drop rows missing key categorical features
    df = df.dropna(subset=['Make', 'Model_Normalized', 'Year'])

    # Label encode categoricals
    le_make     = LabelEncoder()
    le_model    = LabelEncoder()
    le_district = LabelEncoder()

    df['Make_enc']     = le_make.fit_transform(df['Make'].astype(str))
    df['Model_enc']    = le_model.fit_transform(df['Model_Normalized'].astype(str))
    df['District_enc'] = le_district.fit_transform(df['District'].fillna('Unknown').astype(str))

    feature_cols = [
        'Make_enc', 'Model_enc', 'Year', 'vehicle_age',
        'Milleage', 'District_enc', 'pub_month', 'pub_week'
    ]

    X = df[feature_cols]
    y = df['Price']

    encoders = {
        'le_make':     le_make,
        'le_model':    le_model,
        'le_district': le_district,
    }

    return X, y, encoders, df



def train_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

    model = GradientBoostingRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        random_state=42,
        verbose=0
    )
    model.fit(X_train, y_train)

    # Evaluation
    y_pred = model.predict(X_test)
    mae    = mean_absolute_error(y_test, y_pred)
    mape   = mean_absolute_percentage_error(y_test, y_pred) * 100
    r2     = r2_score(y_test, y_pred)

    print("\n=== Model Performance ===")
    print(f"  R²   : {r2:.4f}")
    print(f"  MAE  : LKR {mae:,.0f}")
    print(f"  MAPE : {mape:.1f}%")

    return model, X_test, y_test, y_pred



def compute_time_trends(df: pd.DataFrame, min_weeks: int = 3, max_weekly_change_pct: float = 0.05):
    """
    Fits a linear trend on weekly median prices for each Make+Model.
    Returns a DataFrame with current price, next-week and next-month forecasts.
    """
    df = df.copy()
    df['year_week'] = df['published date'].dt.year * 100 + df['pub_week']

    results = []

    for (make, model), group in df.groupby(['Make', 'Model_Normalized']):
        group = group[(group['Price'] > 100_000) & (group['Price'] < 30_000_000)]
        weekly = (
            group.groupby('year_week')['Price']
            .median()
            .reset_index()
            .sort_values('year_week')
        )

        current_price = group['Price'].median()

        if len(weekly) < min_weeks:
            results.append({
                'Make': make, 'Model': model,
                'Current_Price': round(current_price),
                'Next_Week_Price': round(current_price),
                'Next_Month_Price': round(current_price),
                'Weekly_Trend_LKR': 0,
                'Trend': 'Insufficient data'
            })
            continue

        # Fit linear trend on weekly medians
        X_t = np.arange(len(weekly)).reshape(-1, 1)
        y_t = weekly['Price'].values
        lr  = LinearRegression().fit(X_t, y_t)

        raw_slope = lr.coef_[0]

        # Cap slope at ±5% of current price per week for realism
        cap = current_price * max_weekly_change_pct
        slope = np.clip(raw_slope, -cap, cap)

        next_week_price  = max(current_price + slope * 1, 100_000)
        next_month_price = max(current_price + slope * 4, 100_000)

        if slope > 10_000:
            trend = 'Rising'
        elif slope < -10_000:
            trend = 'Falling'
        else:
            trend = 'Stable'

        results.append({
            'Make': make, 'Model': model,
            'Current_Price': round(current_price),
            'Next_Week_Price': round(next_week_price),
            'Next_Month_Price': round(next_month_price),
            'Weekly_Trend_LKR': round(slope),
            'Week_Change_Pct': round((next_week_price - current_price) / current_price * 100, 2),
            'Month_Change_Pct': round((next_month_price - current_price) / current_price * 100, 2),
            'Trend': trend
        })

    return pd.DataFrame(results)




def predict_single(
    make: str,
    model_name: str,
    year: int,
    mileage: float,
    district: str,
    pub_week: int,
    pub_month: int,
    gbr_model,
    encoders: dict,
    current_year: int = 2026
) -> float:
    """
    Predict the price of a single vehicle.

    Example:
        price = predict_single(
            make='Toyota', model_name='Axio', year=2015,
            mileage=105000, district='Colombo',
            pub_week=8, pub_month=2,
            gbr_model=model, encoders=encoders
        )
    """
    le_make     = encoders['le_make']
    le_model    = encoders['le_model']
    le_district = encoders['le_district']

    def safe_encode(le, value, default=0):
        try:
            return le.transform([str(value)])[0]
        except ValueError:
            print(f"  Warning: '{value}' not seen during training. Using default encoding.")
            return default

    make_enc     = safe_encode(le_make, make)
    model_enc    = safe_encode(le_model, normalize_model_name(model_name))
    district_enc = safe_encode(le_district, district)
    vehicle_age  = current_year - year

    features = [[make_enc, model_enc, year, vehicle_age, mileage, district_enc, pub_month, pub_week]]
    return gbr_model.predict(features)[0]




def save_artifacts(model, encoders: dict, save_dir: str = '.'):
    joblib.dump(model,                  f'{save_dir}/gbr_model.pkl')
    joblib.dump(encoders['le_make'],    f'{save_dir}/le_make.pkl')
    joblib.dump(encoders['le_model'],   f'{save_dir}/le_model.pkl')
    joblib.dump(encoders['le_district'],f'{save_dir}/le_district.pkl')
    print(f"Artifacts saved to '{save_dir}/'")


def load_artifacts(save_dir: str = '.'):
    model = joblib.load(f'{save_dir}/gbr_model.pkl')
    encoders = {
        'le_make':     joblib.load(f'{save_dir}/le_make.pkl'),
        'le_model':    joblib.load(f'{save_dir}/le_model.pkl'),
        'le_district': joblib.load(f'{save_dir}/le_district.pkl'),
    }
    print("Artifacts loaded.")
    return model, encoders



def main(filepath: str, save_dir: str = '.', min_listings: int = 10):
    print("=" * 60)
    print("  Vehicle Price Prediction — Full Pipeline")
    print("=" * 60)

    # Step 1: Load
    df = load_data(filepath)

    # Step 2: Clean model names
    df = clean_model_names(df)

    # Step 3: Feature engineering
    df = engineer_features(df)

    # Step 4: Preprocess & encode
    X, y, encoders, df_clean = preprocess(df)

    # Step 5: Train GBR
    print("\n--- Training Gradient Boosting Regressor ---")
    model, X_test, y_test, y_pred = train_model(X, y)

    # Step 6: Feature importance
    feature_names = [
        'Make', 'Model', 'Year', 'Vehicle Age',
        'Mileage', 'District', 'Pub Month', 'Pub Week'
    ]
    importances = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
    print("\n--- Feature Importances ---")
    print(importances.round(4).to_string())

    # Step 7: Time-series trend for top models
    print("\n--- Computing Time-Series Trends ---")
    top_models = (
        df_clean.groupby(['Make', 'Model_Normalized'])['Price']
        .count()
        .reset_index(name='count')
        .query(f'count >= {min_listings}')
        .sort_values('count', ascending=False)
        .head(20)
    )
    top_model_pairs = set(zip(top_models['Make'], top_models['Model_Normalized']))

    df_top = df_clean[
        df_clean.apply(lambda r: (r['Make'], r['Model_Normalized']) in top_model_pairs, axis=1)
    ]
    trends = compute_time_trends(df_top)
    print(f"Trend predictions generated for {len(trends)} models.")

    print("\n--- Top 10 Trend Predictions ---")
    cols = ['Make', 'Model', 'Current_Price', 'Next_Week_Price', 'Next_Month_Price',
            'Week_Change_Pct', 'Month_Change_Pct', 'Trend']
    print(trends[cols].head(10).to_string(index=False))

    # Step 8: Save
    save_artifacts(model, encoders, save_dir)
    trends.to_csv(f'{save_dir}/trend_predictions.csv', index=False)

    # Step 9: Demo single prediction
    print("\n--- Demo: Single Vehicle Prediction ---")
    demo_price = predict_single(
        make='Toyota', model_name='Axio', year=2015,
        mileage=105000, district='Colombo',
        pub_week=8, pub_month=2,
        gbr_model=model, encoders=encoders
    )
    print(f"  Toyota Axio 2015 (105k km, Colombo) → LKR {demo_price:,.0f}")

    print("\nPipeline complete.")
    return model, encoders, trends




if __name__ == '__main__':

    filepath = 'clean_1__1_.csv'   # your dataset
    save_dir = '.'                 # current folder

    model, encoders, trends = main(filepath, save_dir)

In [ ]:
# ================================
# Sri Lanka Vehicle Price Model
# Full Training + Test Accuracy
# ================================

import pandas as pd
import numpy as np
import re
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

warnings.filterwarnings("ignore")


# -------------------------------
# 1. Load Dataset
# -------------------------------
df = pd.read_csv('/content/drive/MyDrive/ScrapeTraining/clean#1.csv')
print(f"Dataset Loaded: {len(df):,} rows")


# -------------------------------
# 2. Clean Model Names
# -------------------------------
def normalize_model_name(model):
    if pd.isna(model):
        return model
    model = str(model).strip()
    model = re.sub(r'\s+', ' ', model)
    return model.title()

df['Model_Normalized'] = df['Model'].apply(normalize_model_name)


# -------------------------------
# 3. Feature Engineering
# -------------------------------
df['published date'] = pd.to_datetime(df['published date'], errors='coerce')
df['pub_month'] = df['published date'].dt.month
df['pub_week']  = df['published date'].dt.isocalendar().week.astype(int)

CURRENT_YEAR = 2026
df['vehicle_age'] = CURRENT_YEAR - df['Year']


# -------------------------------
# 4. Preprocessing
# -------------------------------

# Remove missing price
df = df.dropna(subset=['Price'])

# Remove extreme price outliers
df = df[(df['Price'] > 100_000) & (df['Price'] < 30_000_000)]

# Fill mileage
df['Milleage'] = df['Milleage'].fillna(
    df.groupby('Model_Normalized')['Milleage'].transform('median')
)
df['Milleage'] = df['Milleage'].fillna(df['Milleage'].median())

# Drop missing key columns
df = df.dropna(subset=['Make', 'Model_Normalized', 'Year'])

# -------------------------------
# 5. Encoding
# -------------------------------
le_make = LabelEncoder()
le_model = LabelEncoder()
le_district = LabelEncoder()

df['Make_enc'] = le_make.fit_transform(df['Make'].astype(str))
df['Model_enc'] = le_model.fit_transform(df['Model_Normalized'].astype(str))
df['District_enc'] = le_district.fit_transform(df['District'].fillna('Unknown').astype(str))


# -------------------------------
# 6. Feature Selection
# -------------------------------
feature_cols = [
    'Make_enc',
    'Model_enc',
    'Year',
    'vehicle_age',
    'Milleage',
    'District_enc',
    'pub_month',
    'pub_week'
]

X = df[feature_cols]
y = df['Price']


# -------------------------------
# 7. Train-Test Split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train Size: {len(X_train)}")
print(f"Test Size: {len(X_test)}")


# -------------------------------
# 8. Train Model
# -------------------------------
model = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)

model.fit(X_train, y_train)


# -------------------------------
# 9. Test Accuracy
# -------------------------------
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100

print("\n==============================")
print("      TEST ACCURACY")
print("==============================")
print(f"R² Score      : {r2:.4f}")
print(f"Accuracy (%)  : {r2*100:.2f}%")
print(f"MAE           : LKR {mae:,.0f}")
print(f"RMSE          : LKR {rmse:,.0f}")
print(f"MAPE          : {mape:.2f}%")
print("==============================")
